# Solve Field

This notebook can be used to determine the WCS solution of a WFI or OmegaCAM image.
It uses astrometry.net solve_field command. It is a necessary step before running scamp which needs a WCS accurate to approx 1", otherwise it fails.

Before start please be sure you can run the solve_field command from the astrometry package by tipying
/usr/local/astrometry/bin/solve-field

or the correspoinding command in your system.

If you get something like
ERROR: You didn't specify any files to process.<br>
This program is part of the Astrometry.net suite.<br>
For details, visit http://astrometry.net.<br>
Git URL https://github.com/dstndstn/astrometry.net<br>
Revision 0.89, date Mon_Jan_17_07:47:26_2022_-0500.<br>

then you are okay.

This notebook will create a independent fits image for each CCD within the input mosaic. Then run solve_field on each one to find an astrometric solution. This solution will be incorporated to a new mosaic which will be use later.
Please be sure you have enough space in your hard disk, as this notebook will effectively created two new mosaic images for each input one.

Tested on:

- linux Ubuntu laptop using the index fields 4100 and 5200 series from astromatic.
- Scientific Linux server
- Macintel laptop running OSC VENTURA

Known issues:
- For some reason, the astrometry package failed during the flow of the script, but running the same command in terminal directly over the CCD fits image retrieves a solution in seconds.
- For those CCDs that have failed at retrieve a solution with astrometry, are copied directly into the final mosaic, causing some shifts between CCDs.

Lientur

In [2]:
import os
import sys

import glob
import logging
import multiprocessing

sys.path.append(os.path.join(os.path.abspath(''), '..')) # to be removed in production
from pycavi import pycavi_solve
from pycavi import pycavi_tools
from pycavi import pycavi_instrument

This will be executed when the package is imported.


In [ ]:
import multiprocessing

multiprocessing.cpu_count()

In [3]:
# Set the reduced science dir where it will look for the reduced science images, same as previous notebooks
# For each target within the folder will check the filters before start to work on each observation.
# In this case, during the tests I consider only the images in the filter NB_659.

config_dir = os.getcwd()+'/'+'../config/'
solve_field_path = "/usr/local/opt/astrometry-net/bin/solve-field"

config = pycavi_tools.read_pycavi_config(config_dir)
instrument = config['Provenance']['instrument'].lower()
reduced_science_dir = config['Directories']['reduced_science_dir']
reduced_std_dir = config['Directories']['reduced_std_dir']
log_dir = config['Directories']['project_logs_dir']

# Should read the instrument configuration file. This ensure
# proper keyword parsing.
#
instrument = pycavi_instrument.Instrument(config_dir=config_dir, instrument=instrument)

# a list with the tags of image filters. During the tests I worked only with r_SDSS and NB_659
filters_to_sol = ["BB_U_38_ESO841", "BB_V_89_ESO843", "BB_B_99_ESO842", "g_SDSS", "r_SDSS", "NB_659", "NB_OIII_8_ESO859"]

# target list will include all the observed targets within the reduced_science directory
target_list = []
for object_ in glob.glob(reduced_science_dir + "*"):
    target_list.append(object_)

for object_ in glob.glob(reduced_std_dir + "*"):
    target_list.append(object_)
    
target_list.sort()

# for each observed target will all the images which filters correspond to the ones given in filters_to_sol
obs_to_solve = []
for target_ in target_list:
    for filter_ in filters_to_sol:
        for observation_ in glob.glob(target_ + "/" + filter_ + "/*"):
            obs_to_solve.append(observation_)

for i in range(len(obs_to_solve)):
    print(i, obs_to_solve[i])


config_dir: /Users/fselman/Software/Pycavi/notebooks/../config/ instrument: omegacam
0 /Users/fselman/Pictures/OCAM/Multi//science/CP_Pup/r_SDSS/2017-03-13T01:57:49
1 /Users/fselman/Pictures/OCAM/Multi//science/M67/g_SDSS/2014-12-02T06:32:53
2 /Users/fselman/Pictures/OCAM/Multi//science/M67/r_SDSS/2014-12-02T06:51:24
3 /Users/fselman/Pictures/OCAM/Multi//science/M67/NB_659/2014-12-02T07:04:51
4 /Users/fselman/Pictures/OCAM/Multi//std/SA101/g_SDSS/2014-12-02T05:57:25
5 /Users/fselman/Pictures/OCAM/Multi//std/SA101/r_SDSS/2014-12-02T05:57:25
6 /Users/fselman/Pictures/OCAM/Multi//std/SA98/NB_659/2014-12-02T07:31:14


In [4]:
# Loop to solve each of the reduced partitions
#
# For testing you might want to select a single partition here
# example: 
#
# For that, select the index from the list displayed in the previous
# cell and add it to the idx_selected list. If this list is empty, then
# all science partitions will be reduced, one at a time.
#
idx_selected = [0]
for i in range(len(obs_to_solve)):

    # skip indices not in list idx_selected
    if not (i in idx_selected or not idx_selected):
        continue

    pycavi_solve.run_py4_solve(obs_to_solve[i], instrument, 
                                           cpu_count=multiprocessing.cpu_count()-2, 
                                           redo_astrometry=False, 
                                           solve_field_path=solve_field_path)

# job done
logging.info("\n\n ****** Py4 Done! ******\n\n")


2024-10-04 12:12:46,869:INFO:current directory: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49
2024-10-04 12:12:46,877:INFO:solving image 01/10: OMEGA.2017-03-13T02:44:33.457_pre.fits
2024-10-04 12:12:46,878:INFO:


Solving field OMEGA.2017-03-13T02:44:33.457_pre
2024-10-04 12:12:46,880:INFO:run_solve_field: directory /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre created 
2024-10-04 12:12:46,936:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__96_solve_field/ Created 
2024-10-04 12:12:47,069:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__65_solve_field/ Created 


OMEGA.*_pre.fits


2024-10-04 12:12:47,191:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__66_solve_field/ Created 
2024-10-04 12:12:47,305:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__67_solve_field/ Created 
2024-10-04 12:12:47,437:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__68_solve_field/ Created 
2024-10-04 12:12:47,550:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__73_solve_field/ Created 
2024-10-04 12:12:47,657:INFO:run_solve_field: solve-field directory /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OME

This will be executed when the package is imported.



****************** solvefield ***************
dir_name2: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__74_solve_field  
file_name2: ESO_CCD__74.fits 
sub_dir_name: ESO_CCD__74_solve_field
solvefield: running solve-field in directory: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__74_solve_field
******************



solvefield: command: /usr/local/opt/astrometry-net/bin/solve-field  --ra 122.975882 --dec -35.313845 --radius=1.0  --scale-units arcsecperpix --scale-low 0.19 --scale-high 0.23 --overwrite --crpix-x  6391 --crpix-y  4124 --no-tweak --no-plots --new-fits ESO_CCD__74_newwcs.fits ESO_CCD__74.fits
run_command: cmdfile: solvefield_command_ESO_CCD__74_solve_field.txt
run_command: logfile: solvefield_command_ESO_CCD__74_solve_field.log
run_command: str_cmd: /usr/local/opt/astrometry

2024-10-04 12:14:25,753:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__90_solve_field//ESO_CCD__90.wcs
2024-10-04 12:14:25,764:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__91_solve_field//ESO_CCD__91.wcs
2024-10-04 12:14:25,776:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__92_solve_field//ESO_CCD__92.wcs
2024-10-04 12:14:25,787:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:44:33.457_pre/ESO_CCD__69_solve_field//ESO_CCD__69.wcs
2024-10-04 12:14:25,797:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:5

...

2024-10-04 12:28:31,461:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__89_solve_field//ESO_CCD__89.wcs
2024-10-04 12:28:31,472:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__90_solve_field//ESO_CCD__90.wcs
2024-10-04 12:28:31,482:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__91_solve_field//ESO_CCD__91.wcs
2024-10-04 12:28:31,493:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__92_solve_field//ESO_CCD__92.wcs
2024-10-04 12:28:31,505:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:5

wcs_fits: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__89_solve_field//ESO_CCD__89_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__90_solve_field//ESO_CCD__90_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__91_solve_field//ESO_CCD__91_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__92_solve_field//ESO_CCD__92_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__69_solve_field//ESO_CCD__69_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__70_solve_field//ESO_

2024-10-04 12:28:31,670:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__94_solve_field//ESO_CCD__94.wcs
2024-10-04 12:28:31,688:INFO:create_wcs_frame: WCS file exists: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__95_solve_field//ESO_CCD__95.wcs
2024-10-04 12:28:31,700:INFO:create_wcs_frame: creating /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/OMEGA.2017-03-13T02:45:38.287_pre_wcs.fits


wcs_fits: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__94_solve_field//ESO_CCD__94_newwcs.fits
wcs_fits: /Users/fselman/Pictures/OCAM/Multi/science/CP_Pup/r_SDSS/2017-03-13T01:57:49/OMEGA.2017-03-13T02:45:38.287_pre/ESO_CCD__95_solve_field//ESO_CCD__95_newwcs.fits


2024-10-04 12:28:36,321:INFO:

 ****** Py4 Done! ******


